In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')
test_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / 'result_v1.csv')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

## 1. 데이터 피쳐링(파생변수 생성)

In [3]:
# ============================================================
# 1-3. 피쳐링 결과: 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 3. 강우사상 라벨링, GroupKFold 5-Fold 배정(CSV로 저장)

In [4]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분: diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

## 4. Tidal, Normal 데이터셋 분할

In [5]:
# ============================================================
# Tidal / Normal 데이터 분리
# ============================================================

# tidal 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 114,111
Tidal      : 2,169
Normal     : 111,942

데이터 분리 검증 완료


## 5. 독립변수 / 종속변수 분리

In [ ]:
# ============================================================
# 5. 독립변수 / 종속변수 분리
# ============================================================

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

# Tidal
X_tidal_train = tidal_train[feature_cols].copy()
X_tidal_test = tidal_test[feature_cols].copy()

y_tidal_train = tidal_train[target_col]

# Normal
X_normal_train = normal_train[feature_cols].copy()
X_normal_test = normal_test[feature_cols].copy()

y_normal_train = normal_train[target_col]

# 데이터 분리 검증
assert X_tidal_train.index.equals(y_tidal_train.index)
assert X_normal_train.index.equals(y_normal_train.index)

assert X_tidal_train.columns.equals(X_tidal_test.columns)
assert X_normal_train.columns.equals(X_normal_test.columns)

print('독립변수 / 종속변수 분리 검증 완료')

독립변수 / 종속변수 분리 검증 완료


## 7. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

In [7]:
# ============================================================
# 7-1. Normal CatBoost 5-Fold 배정 csv 불러와서 매칭
# ============================================================

# 고정 Fold 매핑 파일 불러오기

N_SPLITS = 5

fold_match = pd.read_csv(
    PROJECT_ROOT.parent / 'data' / '0_kf_train_full_match.csv'
)

# event_id 기준으로 Fold 번호 매칭
normal_train['fold_num'] = normal_train['event_id'].map(
    fold_match.set_index('event_id')['fold_num']  # 매핑규칙: 'event_id'로 조인, 'fold_num' 부여
)

# 매칭 검증
assert normal_train['fold_num'].notna().all()
assert normal_train['fold_num'].between(1, N_SPLITS).all()

print('Normal Fold 매칭 완료')

Normal Fold 매칭 완료


In [8]:
# ============================================================
# 7-2. Normal CatBoost 5-Fold OOF 복원 및 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

# Fold별 저장 모델 불러오기 및 예측
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_tr = y_normal_train.loc[train_mask]
    y_kf_val = y_normal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # 저장된 CatBoost Fold별 모델 불러오기
    model = CatBoostRegressor()
    model.load_model(
        PROJECT_ROOT.parent / 'model' / f'cat_{fold}fold_normal.cbm'
    )

    # Fold별 kf_val 예측
    val_pred = model.predict(X_kf_val)

    # Fold별 OOF 예측 및 RMSE
    normal_oof_pred.loc[X_kf_val.index] = val_pred
    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 전체 예측
    test_pred = model.predict(X_normal_test)
    normal_test_preds[f'fold_{fold}'] = test_pred  # Fold별 예측결과가 하나의 컬럼으로 삽입

    # Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': len(X_kf_tr),
        'valid_count': len(X_kf_val),
        'oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
    )


========== Normal Fold 1 ==========
Fold 1 | OOF RMSE=0.233810 | 

========== Normal Fold 2 ==========
Fold 2 | OOF RMSE=0.165635 | 

========== Normal Fold 3 ==========
Fold 3 | OOF RMSE=0.260627 | 

========== Normal Fold 4 ==========
Fold 4 | OOF RMSE=0.210118 | 

========== Normal Fold 5 ==========
Fold 5 | OOF RMSE=0.207717 | 


In [9]:
# ============================================================
# 7-2. Normal CatBoost 학습 및 예측 결과 검증
# ============================================================

# OOF 예측 검증
assert normal_oof_pred.index.equals(y_normal_train.index)
assert normal_oof_pred.notna().all()

# Test 예측 검증
assert normal_test_preds.index.equals(X_normal_test.index)
assert normal_test_preds.shape == (len(X_normal_test), N_SPLITS)
assert normal_test_preds.notna().all().all()

# 전체 OOF RMSE
normal_oof_rmse = root_mean_squared_error(
    y_normal_train, normal_oof_pred
)

print(f'Normal 전체 OOF RMSE: {normal_oof_rmse:.6f}')
print(f'Normal Test 예측: {normal_test_preds.shape}')
print('Normal CatBoost 검증 완료')

Normal 전체 OOF RMSE: 0.217860
Normal Test 예측: (111942, 5)
Normal CatBoost 검증 완료


In [10]:
# ============================================================
# 7-3. Normal CatBoost 5-Fold 앙상블 예측: 이건 test_future용임
# ============================================================

pred_ensemble_normal = normal_test_preds.mean(axis=1)  # axis=1은 행별집계
# 결과물: 하나의 행에 fold별 예측값이 컬럼별로 배치 -> 행별집계 -> 행별 1개 mean값(인덱스 유지)

# 검증
assert pred_ensemble_normal.index.equals(X_normal_test.index)
assert pred_ensemble_normal.notna().all()

print(f'Normal Test: {len(normal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_normal):,}행')
print('Normal 앙상블 예측 검증 완료')

Normal Test: 111,942행
앙상블 예측: 111,942행
Normal 앙상블 예측 검증 완료


In [11]:
# ============================================================
# 7-3. Normal OOF 잔차 생성
# ============================================================

normal_oof_residual = y_normal_train - normal_oof_pred

print(normal_oof_residual.describe())
print(f'잔차 RMSE: {np.sqrt(np.mean(normal_oof_residual**2)):.6f}')

count    735358.000000
mean         -0.001034
std           0.217858
min          -3.099884
25%          -0.047678
50%          -0.012541
75%           0.009384
max           6.841597
dtype: float64
잔차 RMSE: 0.217860


In [12]:
# ============================================================
# 7-4. Normal 잔차 CatBoost 5-Fold 학습 및 예측
# ============================================================

# 결과 저장용 변수
normal_resid_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_resid_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_resid_fold_results = []

# Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Residual Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_tr = normal_oof_residual.loc[train_mask]
    y_kf_val = normal_oof_residual.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 잔차 회귀모델
    model = CatBoostRegressor(
        iterations=1100,
        random_seed=42,
        cat_features=['station_id'],
        thread_count=-1
    )

    model.fit(
        X_kf_tr, y_kf_tr,
        verbose=500
    )

    # Fold별 잔차 OOF 예측
    val_pred = model.predict(X_kf_val)

    normal_resid_oof_pred.loc[X_kf_val.index] = val_pred

    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 잔차 예측
    test_pred = model.predict(X_normal_test)
    normal_resid_test_preds[f'fold_{fold}'] = test_pred

    # Fold 결과 저장
    normal_resid_fold_results.append({
        'fold': fold,
        'train_count': len(X_kf_tr),
        'valid_count': len(X_kf_val),
        'resid_oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'Residual OOF RMSE={fold_oof_rmse:.6f}'
    )


========== Normal Residual Fold 1 ==========
Learning rate set to 0.103781
0:	learn: 0.2132974	total: 323ms	remaining: 5m 55s
500:	learn: 0.1922165	total: 1m 18s	remaining: 1m 33s
1000:	learn: 0.1826325	total: 2m 30s	remaining: 14.9s
1099:	learn: 0.1808859	total: 2m 44s	remaining: 0us
Fold 1 | Residual OOF RMSE=0.266109

========== Normal Residual Fold 2 ==========
Learning rate set to 0.10378
0:	learn: 0.2288533	total: 146ms	remaining: 2m 40s
500:	learn: 0.2077550	total: 1m 12s	remaining: 1m 26s
1000:	learn: 0.1970895	total: 2m 24s	remaining: 14.3s
1099:	learn: 0.1953604	total: 2m 39s	remaining: 0us
Fold 2 | Residual OOF RMSE=0.177015

========== Normal Residual Fold 3 ==========
Learning rate set to 0.103782
0:	learn: 0.2054687	total: 153ms	remaining: 2m 48s
500:	learn: 0.1850421	total: 1m 20s	remaining: 1m 36s
1000:	learn: 0.1750638	total: 2m 39s	remaining: 15.8s
1099:	learn: 0.1733096	total: 2m 54s	remaining: 0us
Fold 3 | Residual OOF RMSE=0.279644

========== Normal Residual Fold

In [13]:
# ============================================================
# 7-5. Normal 잔차모델 성능 및 보정 효과 검증
# ============================================================

assert normal_resid_oof_pred.index.equals(y_normal_train.index)
assert normal_resid_oof_pred.notna().all()

assert normal_resid_test_preds.index.equals(X_normal_test.index)
assert normal_resid_test_preds.shape == (len(X_normal_test), N_SPLITS)
assert normal_resid_test_preds.notna().all().all()

# 잔차모델 OOF RMSE
normal_resid_oof_rmse = root_mean_squared_error(
    normal_oof_residual,
    normal_resid_oof_pred
)

# 기존 Normal OOF 예측에 잔차 예측값 가산
normal_corrected_oof_pred = normal_oof_pred + normal_resid_oof_pred

# 보정 후 Normal OOF RMSE
normal_corrected_oof_rmse = root_mean_squared_error(
    y_normal_train,
    normal_corrected_oof_pred
)

print(f'기존 Normal OOF RMSE: {normal_oof_rmse:.6f}')
print(f'잔차모델 OOF RMSE:    {normal_resid_oof_rmse:.6f}')
print(f'보정 후 OOF RMSE:     {normal_corrected_oof_rmse:.6f}')
print(f'RMSE 변화량:          {normal_corrected_oof_rmse - normal_oof_rmse:+.6f}')

기존 Normal OOF RMSE: 0.217860
잔차모델 OOF RMSE:    0.239082
보정 후 OOF RMSE:     0.239082
RMSE 변화량:          +0.021221


In [14]:
# 기존 Normal Test 예측 + 잔차 보정
normal_corrected_test_pred = (
    normal_test_preds.mean(axis=1)
    + normal_resid_test_preds.mean(axis=1)
)

print(normal_corrected_test_pred.shape)
print(normal_corrected_test_pred.describe())

(111942,)
count    111942.000000
mean          0.104235
std           0.193560
min          -0.683960
25%           0.011168
50%           0.037854
75%           0.116308
max           4.028446
dtype: float64


In [16]:
# ============================================================
# 7-6. Normal Test 잔차 보정 효과 검증
# ============================================================

y_normal_test = normal_test[target_col]

# 기존 Normal Test 예측 (5-Fold 평균)
normal_test_pred = normal_test_preds.mean(axis=1)

# 잔차 보정 후 Normal Test 예측
normal_corrected_test_pred = (
    normal_test_pred + normal_resid_test_preds.mean(axis=1)
)

# Test 실제값과 인덱스 일치 확인
assert normal_test_pred.index.equals(y_normal_test.index)
assert normal_corrected_test_pred.index.equals(y_normal_test.index)

# 보정 전후 RMSE
normal_test_rmse = root_mean_squared_error(
    y_normal_test, normal_test_pred
)

normal_corrected_test_rmse = root_mean_squared_error(
    y_normal_test, normal_corrected_test_pred
)

# 결과 출력
print(f'기존 Normal Test RMSE:    {normal_test_rmse:.6f}')
print(f'보정 후 Normal Test RMSE: {normal_corrected_test_rmse:.6f}')
print(f'RMSE 변화량:              {normal_corrected_test_rmse - normal_test_rmse:+.6f}')
print(f'RMSE 변화율:              {(normal_corrected_test_rmse / normal_test_rmse - 1) * 100:+.2f}%')

기존 Normal Test RMSE:    0.198681
보정 후 Normal Test RMSE: 0.200686
RMSE 변화량:              +0.002005
RMSE 변화율:              +1.01%


## 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성

In [27]:
# ============================================================
# 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성
# ============================================================

# 예측 결과 통합
pred_test = pd.concat([
    pred_ensemble_tidal,
    pred_ensemble_normal
])

# 제출용 데이터프레임 생성
final_result = pred_test.rename('pred_maxrise_6h').reset_index()

# 검증
assert final_result.columns.tolist() == ['row_id', 'pred_maxrise_6h']
assert len(final_result) == len(test)
assert final_result['row_id'].is_unique
assert set(final_result['row_id']) == set(test.index)
assert final_result['pred_maxrise_6h'].notna().all()

print('제출 데이터 생성 및 검증 완료')
print(final_result.shape)

제출 데이터 생성 및 검증 완료
(186263, 2)


In [30]:
final_result.to_csv(
    PROJECT_ROOT / 'data' / 'result_v1.csv',
    index=False,
    encoding='utf-8'
)

In [28]:
final_result.head(100)

,row_id,pred_maxrise_6h
0,TE_000063287,-0.212625
1,TE_000067268,2.738044
2,TE_000049674,0.132557
3,TE_000038339,-0.231214
4,TE_000064879,2.713916
...,...,...
95,TE_000145092,-0.841638
96,TE_000134399,-0.937899
97,TE_000044584,0.464997
98,TE_000128178,1.115125
